In [20]:
# hypotheses

# both heavinly left and heavily right leaning articles will have more extreme sentiments
# they will also have more extreme negative sentiment expressed regarding the opposing perspective

In [21]:
from data_loader import load_split
import pandas as pd
import numpy as np
import nltk

In [22]:
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /Users/sarah/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /Users/sarah/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [23]:
data_dir = "/Users/sarah/Documents/SMU/NLP/Article-Bias-Prediction-main/data"
#data_dir = "C:\\Users\\nia_4\\SMU\\8sem\\nlp\\project\\data"

X_train, y_train = load_split(data_dir, "media", "train")
X_val, y_val = load_split(data_dir, "media", "valid")
X_test, y_test = load_split(data_dir, "media", "test")

print(len(X_train), len(X_val), len(X_test))

KeyboardInterrupt: 

In [ ]:
import feature_extraction_sentiment
from feature_extraction_sentiment import extract_emotionality_features

In [ ]:
def build_feature_matrix(texts):
    return np.array([extract_emotionality_features(t) for t in texts])

In [ ]:
from joblib import Parallel, delayed

# speed up feature extraction with parallel processing
def extract_features_batch(texts: list, n_jobs: int = -1) -> "pd.DataFrame":
    rows = Parallel(n_jobs=n_jobs)(delayed(extract_emotionality_features)(t) for t in texts)
    return pd.DataFrame(rows)

In [ ]:
from sklearn.preprocessing import StandardScaler
# extract features for all splits

X_train_feat = extract_features_batch(X_train)
X_val_feat   = extract_features_batch(X_val)
X_test_feat  = extract_features_batch(X_test)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_feat)
X_val_scaled   = scaler.transform(X_val_feat)
X_test_scaled  = scaler.transform(X_test_feat)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

model = LogisticRegression(max_iter=2000, solver = "saga", random_state=42, verbose=1)
model.fit(X_train_scaled, y_train)

val_preds = model.predict(X_val_scaled)

print(classification_report(y_val, val_preds))

Epoch 1, change: 1
Epoch 2, change: 0.21948133
Epoch 3, change: 0.12551977
Epoch 4, change: 0.065676647
Epoch 5, change: 0.063109231
Epoch 6, change: 0.028949705
Epoch 7, change: 0.016216756
Epoch 8, change: 0.0081553868
Epoch 9, change: 0.0061535566
Epoch 10, change: 0.0035135867
Epoch 11, change: 0.0020488746
Epoch 12, change: 0.00093081714
Epoch 13, change: 0.00082286236
Epoch 14, change: 0.00087574017
Epoch 15, change: 0.0008562062
Epoch 16, change: 0.00085537992
Epoch 17, change: 0.00084859133
Epoch 18, change: 0.00085323153
Epoch 19, change: 0.0008553713
Epoch 20, change: 0.00084771729
Epoch 21, change: 0.00084673607
Epoch 22, change: 0.00084552909
Epoch 23, change: 0.00084246563
Epoch 24, change: 0.00083957155
Epoch 25, change: 0.00083716793
Epoch 26, change: 0.00083531015
Epoch 27, change: 0.00083273054
Epoch 28, change: 0.00083012477
Epoch 29, change: 0.00082774204
Epoch 30, change: 0.00082552586
Epoch 31, change: 0.00082360456
Epoch 32, change: 0.00082058925
Epoch 33, change:

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight

weights = compute_sample_weight(class_weight='balanced', y=y_train)

model_gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3)
model_gb.fit(X_train_scaled, y_train, sample_weight=weights)

y_preds_gb = model_gb.predict(X_val_scaled)

print(classification_report(y_val, y_preds_gb))

              precision    recall  f1-score   support

           0       0.37      0.06      0.10      1640
           1       0.43      0.45      0.44       618
           2       0.04      0.59      0.08        98

    accuracy                           0.18      2356
   macro avg       0.28      0.37      0.20      2356
weighted avg       0.37      0.18      0.19      2356



In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train_scaled, y_train)

y_preds_rf = clf.predict(X_val_scaled)

print(classification_report(y_val, y_preds_rf))

              precision    recall  f1-score   support

           0       0.42      0.09      0.14      1640
           1       0.39      0.20      0.27       618
           2       0.04      0.71      0.08        98

    accuracy                           0.14      2356
   macro avg       0.28      0.33      0.16      2356
weighted avg       0.39      0.14      0.17      2356



In [35]:
from sklearn.naive_bayes import GaussianNB

model = GaussianNB()
model.fit(X_train_scaled, y_train)

nb_preds_tfidf = model.predict(X_val_scaled)
print(classification_report(y_val, nb_preds_tfidf))

              precision    recall  f1-score   support

           0       0.64      0.32      0.43      1640
           1       0.25      0.37      0.30       618
           2       0.08      0.47      0.13        98

    accuracy                           0.34      2356
   macro avg       0.32      0.39      0.29      2356
weighted avg       0.52      0.34      0.38      2356



In [19]:
X_train_feat.head()

,vader_compound,vader_pos,vader_neg,vader_neu,vader_sent_mean,vader_sent_std,vader_sent_max,vader_sent_min,exclamation_ratio,question_ratio,...,nrc_fear,nrc_anger,nrc_trust,nrc_surprise,nrc_sadness,nrc_disgust,nrc_joy,nrc_anticipation,nrc_positive,nrc_negative
0,-0.9871,0.089,0.103,0.808,-0.033343,0.451420,0.9428,-0.9118,0.000700,0.0,...,0.037037,0.000000,0.203704,0.037037,0.018519,0.000000,0.166667,0.055556,0.370370,0.111111
1,-0.9542,0.050,0.080,0.870,-0.123158,0.461800,0.7506,-0.8834,0.000000,0.0,...,0.153846,0.043956,0.120879,0.043956,0.142857,0.043956,0.010989,0.087912,0.164835,0.186813
2,0.9988,0.122,0.062,0.816,0.143974,0.477579,0.9136,-0.8755,0.000604,0.0,...,0.081818,0.118182,0.118182,0.063636,0.090909,0.027273,0.081818,0.081818,0.190909,0.145455
3,-0.7227,0.028,0.045,0.927,-0.036608,0.389573,0.7579,-0.8402,0.000000,0.0,...,0.105263,0.105263,0.184211,0.039474,0.026316,0.026316,0.026316,0.092105,0.263158,0.131579
4,0.9982,0.101,0.043,0.856,0.193516,0.428611,0.9158,-0.8689,0.001385,0.0,...,0.087500,0.062500,0.137500,0.037500,0.075000,0.037500,0.075000,0.100000,0.262500,0.125000


In [24]:
test_features = extract_emotionality_features(X_train[0])
print("Number of features:", len(test_features))
print("Feature values:", test_features)

Number of features: 22
Feature values: {'vader_compound': -0.9871, 'vader_pos': 0.089, 'vader_neg': 0.103, 'vader_neu': 0.808, 'vader_sent_mean': np.float64(-0.03334333333333332), 'vader_sent_std': np.float64(0.4514203423147383), 'vader_sent_max': 0.9428, 'vader_sent_min': -0.9118, 'exclamation_ratio': 0.0006997900629811056, 'question_ratio': 0.0, 'all_caps_ratio': 0.0006997900629811056, 'subjectivity': 0.4386586554785086, 'nrc_fear': 0.037037037037037035, 'nrc_anger': 0.0, 'nrc_trust': 0.2037037037037037, 'nrc_surprise': 0.037037037037037035, 'nrc_sadness': 0.018518518518518517, 'nrc_disgust': 0.0, 'nrc_joy': 0.16666666666666666, 'nrc_anticipation': 0.05555555555555555, 'nrc_positive': 0.37037037037037035, 'nrc_negative': 0.1111111111111111}


In [46]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

tfidf = TfidfVectorizer(
    analyzer="word",       # word n-grams for semantics
    ngram_range=(1, 3),
    max_features=10000,   
    min_df=5,
    sublinear_tf=True  
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf   = tfidf.transform(X_val)
X_test_tfidf  = tfidf.transform(X_test)

svd = TruncatedSVD(n_components=300, random_state=42)
X_train_tfidf = svd.fit_transform(X_train_tfidf)
X_val_tfidf   = svd.transform(X_val_tfidf)
X_test_tfidf  = svd.transform(X_test_tfidf)

# Combine with sentiment features
X_train_combined = np.hstack([X_train_tfidf, X_train_scaled])
X_val_combined   = np.hstack([X_val_tfidf,   X_val_scaled])
X_test_combined  = np.hstack([X_test_tfidf,  X_test_scaled])

In [47]:
# TEST TF-IDF results with Logistic Regression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

model = LogisticRegression(class_weight='balanced', max_iter=2000, solver = "saga", random_state=42, verbose=1)
model.fit(X_train_combined, y_train)

lr_preds_tfidf = model.predict(X_val_combined)

print(classification_report(y_val, lr_preds_tfidf))

Epoch 1, change: 1
Epoch 2, change: 0.39778315
Epoch 3, change: 0.24465289
Epoch 4, change: 0.17892167
Epoch 5, change: 0.14117946
Epoch 6, change: 0.11635848
Epoch 7, change: 0.10037038
Epoch 8, change: 0.087619187
Epoch 9, change: 0.077643582
Epoch 10, change: 0.069320412
Epoch 11, change: 0.062577844
Epoch 12, change: 0.056803165
Epoch 13, change: 0.052008412
Epoch 14, change: 0.047683962
Epoch 15, change: 0.044040534
Epoch 16, change: 0.040737634
Epoch 17, change: 0.037709054
Epoch 18, change: 0.03507021
Epoch 19, change: 0.032683395
Epoch 20, change: 0.030533871
Epoch 21, change: 0.028674553
Epoch 22, change: 0.026880544
Epoch 23, change: 0.025303849
Epoch 24, change: 0.023889914
Epoch 25, change: 0.022558362
Epoch 26, change: 0.021379093
Epoch 27, change: 0.02029341
Epoch 28, change: 0.019274025
Epoch 29, change: 0.018343116
Epoch 30, change: 0.017451004
Epoch 31, change: 0.016647323
Epoch 32, change: 0.015925086
Epoch 33, change: 0.015231818
Epoch 34, change: 0.014577631
Epoch 3

In [51]:
from sklearn.naive_bayes import GaussianNB
from sklearn.utils.class_weight import compute_sample_weight

sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

model_nb = GaussianNB()
model_nb.fit(X_train_combined, y_train, sample_weight=sample_weights)

nb_preds_tfidf = model_nb.predict(X_val_combined)
print(classification_report(y_val, nb_preds_tfidf))

              precision    recall  f1-score   support

           0       0.63      0.41      0.50      1640
           1       0.48      0.27      0.35       618
           2       0.04      0.42      0.08        98

    accuracy                           0.38      2356
   macro avg       0.38      0.37      0.31      2356
weighted avg       0.57      0.38      0.44      2356



In [ ]:
# SAVE FEATURES FOR LATER!!

import numpy as np
import pandas as pd
import joblib

pd.DataFrame(X_train_features).to_csv('X_train_features.csv', index=False)
pd.DataFrame(X_val_features).to_csv('X_val_features.csv', index=False)
pd.DataFrame(y_train, columns=['label']).to_csv('y_train.csv', index=False)
pd.DataFrame(y_val, columns=['label']).to_csv('y_val.csv', index=False)

joblib.dump(model, 'sentiment_model.pkl')


In [53]:
val_probs = model_nb.predict_proba(X_val_combined)
test_probs = model_nb.predict_proba(X_test_combined)

import numpy as np

np.save("sentiment_val_probs.npy", val_probs)
np.save("sentiment_test_probs.npy", test_probs)
np.save("sentiment_val_labels.npy", y_val)